# CNN serialization, inference and monitoring

**Learning objective:** Save and reload a CNN, measure inference latency and detect a simple brightness distribution shift.

This notebook is part of the TensorFlow/Keras learning track. It is designed to be read top-to-bottom: intuition → shapes → mathematics → TensorFlow implementation → observed result → interpretation.

> GitHub renders the committed executed output as a static learning artifact. Clone the repository and rerun it in Jupyter/VS Code for live experimentation.


In [1]:
import os, warnings, random
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "bundled with TensorFlow")
print("Execution device(s):", [d.device_type for d in tf.config.list_logical_devices()])


2026-09-21 07:16:35.160258: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789974995.176728    4344 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789974995.181308    4344 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.1
Keras: 3.15.1
Execution device(s): ['CPU']


2026-09-21 07:16:36.929126: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
model=tf.keras.Sequential([tf.keras.layers.Input((28,28,1)),tf.keras.layers.Conv2D(8,3,activation="relu"),tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dense(10,activation="softmax")]); _=model(tf.zeros((1,28,28,1)))
path=Path("CNN")/"artifacts_demo_cnn.keras"; model.save(path); restored=tf.keras.models.load_model(path)
sample=tf.random.uniform((32,28,28,1),seed=SEED); import time; start=time.perf_counter(); out=restored.predict(sample,verbose=0); ms=(time.perf_counter()-start)*1000/len(sample)
print("saved/reloaded:",path,"output:",out.shape,"approx ms/image:",round(ms,3))


saved/reloaded: CNN/artifacts_demo_cnn.keras output: (32, 10) approx ms/image: 2.513


In [3]:
(x,_),_=tf.keras.datasets.fashion_mnist.load_data(); ref=x[:2000].astype("float32")/255; shifted=np.clip(ref*.65,0,1)
stats=pd.DataFrame({"population":["reference","darkened production simulation"],"pixel_mean":[ref.mean(),shifted.mean()],"pixel_std":[ref.std(),shifted.std()]}); display(stats.round(4))


,population,pixel_mean,pixel_std
0,reference,0.2839,0.3535
1,darkened production simulation,0.1846,0.2298


Production image monitoring can track intensity/channel statistics, resolution, corruption rates, class mix, confidence, latency and labeled performance. Drift alarms should trigger investigation, not automatic retraining without validation.
